In [ ]:
# Cell 1: Install Dependencies
!pip install --upgrade awswrangler xgboost scikit-learn pandas numpy

In [ ]:
# Cell 2: Imports & Global Configuration
import os
import pandas as pd
import numpy as np
import xgboost as xgb
import awswrangler as wr
from sklearn.metrics import mean_absolute_error, mean_squared_error
from datetime import datetime

# Update S3_BUCKET and GOLD_DATABASE with your actual resource names
S3_BUCKET = "online-retail-inventory-project"  
GOLD_DATABASE = "gold_databse"             
TABLE_NAME = "product_demand_features"

MODEL_VERSION = "v1.0.0"
EVALUATION_S3_PATH = f"s3://{S3_BUCKET}/ml/evaluation/model_metrics.parquet"
PREDICTIONS_S3_PATH = f"s3://{S3_BUCKET}/ml/predictions/demand_predictions.parquet"
SUMMARY_PREDICTIONS_S3_PATH = f"s3://{S3_BUCKET}/ml/predictions/product_demand_summary.parquet"
MODEL_ARTIFACT_S3_PATH = f"s3://{S3_BUCKET}/ml/models/xgb_model_{MODEL_VERSION}.json"
ATHENA_TEMP_PATH = f"s3://{S3_BUCKET}/athena_temp/"

In [ ]:
# Cell 3: Fetch Data from Athena & Standardize Columns
print(f"[{datetime.now().strftime('%H:%M:%S')}] Querying Gold features from Athena ({GOLD_DATABASE}.{TABLE_NAME})...")

query = f"""
SELECT 
    stockcode,
    transaction_date,
    units_sold,
    sales_lag_1,
    sales_lag_7,
    sales_lag_14,
    sales_lag_28,
    rolling_mean_7,
    rolling_mean_28,
    rolling_std_7,
    day_of_week,
    month,
    is_weekend
FROM "{GOLD_DATABASE}"."{TABLE_NAME}"
WHERE sales_lag_28 IS NOT NULL
"""

df = wr.athena.read_sql_query(
    sql=query, 
    database=GOLD_DATABASE,
    s3_output=ATHENA_TEMP_PATH
)

# Standardize all column names to lowercase (fixes Glue catalog naming issues)
df.columns = df.columns.str.lower()

# Datatype casting and sorting
df['transaction_date'] = pd.to_datetime(df['transaction_date'])
df = df.sort_values(['stockcode', 'transaction_date']).reset_index(drop=True)

FEATURES = [
    'sales_lag_1', 'sales_lag_7', 'sales_lag_14', 'sales_lag_28',
    'rolling_mean_7', 'rolling_mean_28', 'rolling_std_7',
    'day_of_week', 'month', 'is_weekend'
]
TARGET = 'units_sold'

for col in FEATURES:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
df[TARGET] = pd.to_numeric(df[TARGET], errors='coerce').fillna(0)

df.head()

In [ ]:
# Cell 4: Chronological Train / Test Split
max_date = df['transaction_date'].max()
split_date = max_date - pd.Timedelta(days=30)

train_df = df[df['transaction_date'] <= split_date].copy()
test_df = df[df['transaction_date'] > split_date].copy()

X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_test, y_test = test_df[FEATURES], test_df[TARGET]

print(f"Dataset Split | Train Records: {len(train_df):,} | Test Records: {len(test_df):,}")

In [ ]:
# Cell 5: Define Evaluation Metrics Helper Function
def compute_metrics(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
    total_actual = np.sum(y_true)
    wape = (np.sum(np.abs(y_true - y_pred)) / total_actual) * 100.0 if total_actual > 0 else 0.0
    bias = np.mean(y_true - y_pred)
    
    return {
        "model_version": MODEL_VERSION,
        "model_name": model_name,
        "train_window": f"{train_df['transaction_date'].min().strftime('%Y-%m-%d')} to {split_date.strftime('%Y-%m-%d')}",
        "validation_window": f"{(split_date + pd.Timedelta(days=1)).strftime('%Y-%m-%d')} to {max_date.strftime('%Y-%m-%d')}",
        "mae": round(float(mae), 4),
        "rmse": round(float(rmse), 4),
        "wape_pct": round(float(wape), 2),
        "bias": round(float(bias), 4),
        "evaluated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

In [ ]:
# Cell 6: Train Baseline Models & XGBoost Regressor
print(f"[{datetime.now().strftime('%H:%M:%S')}] Training Baseline and XGBoost Models...")

# 1. Naive Baseline (7-Day Lag)
naive_preds = np.clip(test_df['sales_lag_7'].values, 0, None)
naive_metrics = compute_metrics(y_test, naive_preds, "Naive Baseline (7-Day Lag)")

# 2. 7-Day Moving Average Baseline
ma_preds = np.clip(test_df['rolling_mean_7'].values, 0, None)
ma_metrics = compute_metrics(y_test, ma_preds, "7-Day Moving Average")

# 3. XGBoost Regressor
xgb_model = xgb.XGBRegressor(
    n_estimators=200,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective='reg:squarederror'
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=False
)

xgb_preds = np.clip(xgb_model.predict(X_test), 0, None)
xgb_metrics = compute_metrics(y_test, xgb_preds, "XGBoost Regressor")

In [ ]:
# Cell 7: Display Model Performance Comparison
metrics_list = [naive_metrics, ma_metrics, xgb_metrics]
metrics_df = pd.DataFrame(metrics_list)

print("\n=================== MODEL EVALUATION METRICS ===================")
metrics_df[['model_name', 'mae', 'rmse', 'wape_pct', 'bias']]

In [ ]:
# Cell 8: Build Batch Forecast Output Dataset
test_df['predicted_units'] = np.round(xgb_preds, 2)
test_df['model_version'] = MODEL_VERSION
test_df['generated_at'] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

forecast_output_df = test_df[[
    'stockcode', 
    'transaction_date', 
    'units_sold', 
    'predicted_units', 
    'model_version', 
    'generated_at'
]].rename(columns={
    'stockcode': 'product_id',
    'units_sold': 'actual_units'
})

forecast_output_df.head()

In [ ]:
# Cell 9: Compute D7 and D30 Forecast Horizons
print(f"[{datetime.now().strftime('%H:%M:%S')}] Aggregating D7 and D30 forecast horizons...")

forecast_summary_df = forecast_output_df.groupby('product_id').apply(
    lambda x: pd.Series({
        'forecast_d7_units': np.round(x.head(7)['predicted_units'].sum(), 2),
        'forecast_d30_units': np.round(x.head(30)['predicted_units'].sum(), 2),
        'avg_daily_forecast_units': np.round(x.head(30)['predicted_units'].mean(), 2),
        'model_version': MODEL_VERSION,
        'generated_at': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    })
).reset_index()

forecast_summary_df.head()

In [ ]:
# Cell 10: Persist Metrics, Forecasts & Model Artifact to S3
print(f"[{datetime.now().strftime('%H:%M:%S')}] Writing evaluation metrics to {EVALUATION_S3_PATH}...")
wr.s3.to_parquet(df=metrics_df, path=EVALUATION_S3_PATH, mode="overwrite")

print(f"[{datetime.now().strftime('%H:%M:%S')}] Writing demand predictions to {PREDICTIONS_S3_PATH}...")
wr.s3.to_parquet(df=forecast_output_df, path=PREDICTIONS_S3_PATH, mode="overwrite")

print(f"[{datetime.now().strftime('%H:%M:%S')}] Writing summary predictions to {SUMMARY_PREDICTIONS_S3_PATH}...")
wr.s3.to_parquet(df=forecast_summary_df, path=SUMMARY_PREDICTIONS_S3_PATH, mode="overwrite")

# Save model file locally and upload to S3
local_model_path = "xgb_model.json"
xgb_model.save_model(local_model_path)
wr.s3.upload(local_file=local_model_path, path=MODEL_ARTIFACT_S3_PATH)

print(f"\n[{datetime.now().strftime('%H:%M:%S')}] SUCCESS: ML Phase Completed! All artifacts saved to S3.")